# 09 · The optimization ladder

Notebooks 01–06 each removed one bottleneck. This one runs all of them at a single large shape and asks whether the improvements actually compose — or whether, as often happens, each one only helped in the setting where it was introduced.

## 1. The question

Six kernels, one shape (`mnk=4096,4096,4096`), each adding one structural change:

| Step | Kernel | Change |
|---|---|---|
| 0 | gemm 0x | Baseline pipelined GEMM |
| 1 | gemm 1x | 2-CTA MMA + TMA multicast |
| 2 | gemm 2 | Warp specialization |
| 3 | gemm 3 | Static persistent scheduling |
| 4 | gemm 3_1 | Dynamic CLC scheduling |
| 5 | gemm 4 | Preferred + fallback clusters |

**Why this shape:** K=4096 is where notebook 03 showed the latency-hiding wins actually pay. At K=512 several of these steps look like regressions.

### Setup

Every notebook starts here. If this machine cannot profile, the collection cells below print their command instead of failing, and the notebook still reads end to end.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "tools"))
import check_env, nbhelp, trace_summary as ts

env = check_env.report()

Environment
----------------------------------------------------------
  GPU           NVIDIA B300 SXM6 AC  [SM103 (Blackwell)]
  Driver        610.57.04
  cutlass-dsl   4.7.1
  iket API      yes
  run-iket      yes
----------------------------------------------------------
  Live profiling: AVAILABLE


## 2. The instrumentation

None new. Every kernel here was instrumented in an earlier notebook, and the ladder is assembled by comparing warp-lifetime spans across the six traces.

That reuse is the point: consistent instrumentation across variants is what makes them comparable at all.

## 3. Run it

Six runs at a large shape — noticeably slower than the earlier notebooks. Strictly sequential, or the comparison is meaningless.

In [2]:
LADDER = [
    ("gemm0x",  "fp16_gemm_0x_iket_ps_2_detailed.py"),
    ("gemm1x",  "fp16_gemm_1x_iket_detailed.py"),
    ("gemm2",   "fp16_gemm_2_iket_detailed_marks.py"),
    ("gemm3",   "fp16_gemm_3_iket_detailed_marks.py"),
    ("gemm3_1", "fp16_gemm_3_1_iket_detailed_marks.py"),
    ("gemm4",   "fp16_gemm_4_iket_detailed.py"),
]
for tag, src in LADDER:
    cmd = (
        f"run-iket --output-dir out/nb09_{tag} --clobber profile --postprocess all -- "
        f"python {nbhelp.kernel('instrumented', src)} --mnk 4096,4096,4096"
    )
    print(f"===== {tag} =====")
    check_env.run_or_show(cmd, env, est="2-4 min each, ~20 min total")

===== gemm0x =====
$ run-iket --output-dir out/nb09_gemm0x --clobber profile --postprocess all -- python /path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_0x_iket_ps_2_detailed.py --mnk 4096,4096,4096

[run-iket] ============= Dry run application first to collect some info... =============


Running Blackwell fp16 GEMM example 0 with:
  mnk:       [4096, 4096, 4096]
  tolerance: 0.1

a: tensor([[ 1., -2., -2.,  ..., -2., -2., -2.],
        [-1., -2.,  0.,  ...,  1., -1., -1.],
        [-2.,  0., -1.,  ...,  0.,  1., -2.],
        ...,
        [-1., -2.,  0.,  ..., -1., -2.,  1.],
        [-2.,  1., -2.,  ..., -2.,  1.,  0.],
        [-1., -1., -1.,  ..., -1.,  1., -2.]], device='cuda:0',
       dtype=torch.float16)
b: tensor([[-2., -1., -2.,  ...,  1., -1.,  1.],
        [-1.,  1., -2.,  ...,  0.,  0., -1.],
        [-1., -1., -2.,  ...,  1.,  1.,  0.],
        ...,
        [-1., -1., -1.,  ...,  0., -1., -1.],
        [ 0., -1.,  1.,  ...,  0., -2., -1.],
        [ 0.,  0., -2.,  ...,  0.,  1.,  0.]], device='cuda:0',
       dtype=torch.float16)
c: tensor([[1081., 1063.,  948.,  ...,  958., 1132., 1011.],
        [ 873.,  872., 1108.,  ..., 1049., 1259., 1091.],
        [1022.,  818., 1159.,  ...,  936.,  999., 1187.],
        ...,
        [ 920., 1033.,  995.,  ..., 1130

Running Blackwell fp16 GEMM example 0 with:
  mnk:       [4096, 4096, 4096]
  tolerance: 0.1

a: tensor([[ 1., -2., -2.,  ..., -2., -2., -2.],
        [-1., -2.,  0.,  ...,  1., -1., -1.],
        [-2.,  0., -1.,  ...,  0.,  1., -2.],
        ...,
        [-1., -2.,  0.,  ..., -1., -2.,  1.],
        [-2.,  1., -2.,  ..., -2.,  1.,  0.],
        [-1., -1., -1.,  ..., -1.,  1., -2.]], device='cuda:0',
       dtype=torch.float16)
b: tensor([[-2., -1., -2.,  ...,  1., -1.,  1.],
        [-1.,  1., -2.,  ...,  0.,  0., -1.],
        [-1., -1., -2.,  ...,  1.,  1.,  0.],
        ...,
        [-1., -1., -1.,  ...,  0., -1., -1.],
        [ 0., -1.,  1.,  ...,  0., -2., -1.],
        [ 0.,  0., -2.,  ...,  0.,  1.,  0.]], device='cuda:0',
       dtype=torch.float16)
c: tensor([[1081., 1063.,  948.,  ...,  958., 1132., 1011.],
        [ 873.,  872., 1108.,  ..., 1049., 1259., 1091.],
        [1022.,  818., 1159.,  ...,  936.,  999., 1187.],
        ...,
        [ 920., 1033.,  995.,  ..., 1130

[run-iket] Dumped perfetto trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm0x/iket_pid_0x3922af.pftrace
[run-iket] Dumped json trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm0x/iket_pid_0x3922af.trace.json


[run-iket] Dumped compressed trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm0x/iket_pid_0x3922af.pftrace.gz
[run-iket] Dumped perfetto HTML viewer to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm0x/iket_pid_0x3922af.html
[run-iket] To view: cd /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm0x && python3 -m http.server 8080 , then open http://localhost:8080/iket_pid_0x3922af.html
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm0x/tracker
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm0x/gen-config
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm0x/iket
===== gemm1x =====
$ run-iket --output-dir out/nb09_gemm1x --clobber profile --postprocess all -- python /path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_1

...

[run-iket] ============= Dry run application first to collect some info... =============


Running Blackwell fp16 GEMM example 4 (with MIX CGA support):
  mnk:                        [4096, 4096, 4096]
  tolerance:                  0.1
  Preferred cluster shape:    (2, 4, 1)
  Fallback cluster shape:     (2, 1, 1)
  CTA tile shape (MNK):       (256, 256, 64)

/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_4_iket_detailed.py:841: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  preferred_tile_sched_params = utils.ClcDynamicPersistentTileSchedulerParams(
/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_4_iket_detailed.py:859: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  fallback_tile_sched_params = utils.ClcDynamicPersistentTileSchedulerParams(
Verifying reference result...
PASS
[run-iket] Auto-computed context bu

Running Blackwell fp16 GEMM example 4 (with MIX CGA support):
  mnk:                        [4096, 4096, 4096]
  tolerance:                  0.1
  Preferred cluster shape:    (2, 4, 1)
  Fallback cluster shape:     (2, 1, 1)
  CTA tile shape (MNK):       (256, 256, 64)

/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_4_iket_detailed.py:841: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  preferred_tile_sched_params = utils.ClcDynamicPersistentTileSchedulerParams(
/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_4_iket_detailed.py:859: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  fallback_tile_sched_params = utils.ClcDynamicPersistentTileSchedulerParams(
Verifying reference result...
PASS


[run-iket] Dumped perfetto trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm4/iket_pid_0x392529.pftrace
[run-iket] Dumped json trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm4/iket_pid_0x392529.trace.json


[run-iket] Dumped compressed trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm4/iket_pid_0x392529.pftrace.gz
[run-iket] Dumped perfetto HTML viewer to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm4/iket_pid_0x392529.html
[run-iket] To view: cd /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm4 && python3 -m http.server 8080 , then open http://localhost:8080/iket_pid_0x392529.html
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm4/tracker
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm4/gen-config
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb09_gemm4/iket


## 4. The trace

In [3]:
nbhelp.figure("nb09/gemm_ladder_speedup.png", "Each structural step removes a bottleneck. The climb is monotonic to step 3.")

In [4]:
import pathlib
rows, base = [], None
for tag, _ in LADDER:
    d = pathlib.Path(f"out/nb09_{tag}")
    if not d.exists():
        continue
    s = ts.summarize(ts.load(ts.find_trace(d)))
    base = base or s.span_us
    rows.append((tag, f"{s.span_us:.2f} us", f"{base / s.span_us:.2f}x"))
if rows:
    nbhelp.table(rows, ("kernel", "span", "vs baseline"))
else:
    print("Not collected locally — reference numbers below.")

kernel   span       vs baseline
-------  ---------  -----------
gemm0x   113.06 us  1.00x      
gemm1x   92.13 us   1.23x      
gemm2    87.33 us   1.29x      
gemm3    77.25 us   1.46x      
gemm3_1  78.11 us   1.45x      
gemm4    79.17 us   1.43x      


## 5. The finding

Measured on a NVIDIA B300 at `mnk=4096,4096,4096`, warp-lifetime span:

| Step | Kernel | Span | Cumulative |
|---|---|---|---|
| 0 | gemm 0x baseline | 113.31 µs | 1.00× |
| 1 | gemm 1x — 2-CTA + multicast | 92.32 µs | 1.23× |
| 2 | gemm 2 — warp specialization | 87.58 µs | 1.29× |
| 3 | gemm 3 — static persistent | **76.51 µs** | **1.48×** |
| 4 | gemm 3_1 — dynamic CLC | 77.50 µs | 1.46× |
| 5 | gemm 4 — preferred clusters | 78.69 µs | 1.44× |


**113.31 → 76.51 µs: a 1.48× speedup, and steps 0–3 are each faster than the last.**

**Step 1 is the vindication of notebook 03.** The same 2-CTA MMA that was 10% *slower* at K=512 is the single largest step here, worth 1.23× on its own. Nothing about the kernel changed — only the shape it runs at.

**Steps 4 and 5 do not improve on step 3, and that is consistent.** Notebook 05 measured CLC's overhead at 15% on balanced load; notebook 06 found cluster barriers already free. Neither had anything left to win here. They are not failures — they are options whose regimes this shape does not enter, exactly as their own notebooks predicted.

### Why the ladder works

The mechanism behind the climb is consistent across every step: **the wins come from feeding the tensor cores, not from doing more math.**

> **Note on this section.** The metrics below are hardware counters, not IKET output, and this notebook does not collect them.
> To collect them yourself, use Nsight Compute:
> ```
> ncu --set full --section SpeedOfLight python <kernel>.py --mnk 4096,4096,4096
> ```
> Everything above this section is from the IKET traces you just collected.

- DRAM speed-of-light falls from ~40% to ~17% across the ladder — each step removes or hides memory traffic
- The tensor (MMA) pipe holds around 50% throughout — the compute was never the limiter
- Warp specialization cuts inter-warp dependency stalls roughly 3×
- Multicast cuts L2 *read volume* about 20%, while L2 hit rate stays flat around 85% — the win is fewer reads, not better caching

**The transferable point:** IKET told you *where* the time went at each step. Hardware metrics tell you *why*. Neither alone would have explained the ladder.

**Next:** [10 · FMHA — a multi-role pipeline](10_fmha_multirole.ipynb)

---
*Measured on a single NVIDIA B300, public `nvidia-cutlass-dsl` 4.7.1. Your absolute numbers will differ; the shapes in the trace are the transferable part.*